In [1]:
import matplotlib.pyplot as plt
import os

def save_output(dataframes, labels, filepath, heading=""):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    n = len(dataframes)
    heights = [max(2, len(df) * 0.5 + 1) for df in dataframes]
    fig, axes = plt.subplots(n, 1, figsize=(16, sum(heights)))
    if n == 1:
        axes = [axes]
    if heading:
        fig.suptitle(heading, fontsize=12, fontweight='bold')
    for ax, df, label in zip(axes, dataframes, labels):
        ax.axis('off')
        ax.set_title(label, fontsize=10, loc='left', pad=6)
        t = ax.table(cellText=df.astype(str).values, colLabels=df.columns, loc='center', cellLoc='center')
        t.auto_set_font_size(False)
        t.set_fontsize(8)
        t.auto_set_column_width(col=list(range(len(df.columns))))
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved:", filepath)


In [2]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_style('darkgrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'axes.titlesize': 12})

db_path = r"d:\project\DataScience\Data Vortex\social_engine.db"
if os.path.exists(db_path):
    os.remove(db_path)

df_users = pd.read_csv(r"d:\project\DataScience\Data Vortex\Dataset\Social_Engine_Users_Cleaned.csv")
df_posts = pd.read_csv(r"d:\project\DataScience\Data Vortex\Dataset\Social_Engine_Posts_Cleaned.csv")

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS posts;
DROP TABLE IF EXISTS users;

CREATE TABLE users (
    user_id         TEXT PRIMARY KEY,
    location        TEXT,
    language        TEXT,
    account_created TEXT,
    follower_count  INTEGER
);

CREATE TABLE posts (
    post_id              TEXT PRIMARY KEY,
    user_id              TEXT,
    platform             TEXT,
    text_content         TEXT,
    timestamp            TEXT,
    likes                INTEGER,
    shares               INTEGER,
    comments             INTEGER,
    likes_is_outlier     INTEGER,
    shares_is_outlier    INTEGER,
    comments_is_outlier  INTEGER,
    user_in_registry     INTEGER,
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
""")

conn.commit()
df_users.to_sql("users", conn, if_exists="replace", index=False)
df_posts.to_sql("posts", conn, if_exists="replace", index=False)
conn.commit()

print("Tables loaded.")
print(f"Users  : {pd.read_sql('SELECT COUNT(*) AS n FROM users', conn).iloc[0,0]}")
print(f"Posts  : {pd.read_sql('SELECT COUNT(*) AS n FROM posts', conn).iloc[0,0]}")


Tables loaded.
Users  : 1500
Posts  : 12360


In [3]:
cursor.executescript("""
DROP VIEW IF EXISTS vw_post_engagement;
DROP VIEW IF EXISTS vw_user_performance;
DROP VIEW IF EXISTS vw_platform_summary;

CREATE VIEW vw_post_engagement AS
SELECT
    p.post_id,
    p.user_id,
    p.platform,
    p.timestamp,
    SUBSTR(p.timestamp, 1, 7)              AS year_month,
    SUBSTR(p.timestamp, 1, 10)             AS date,
    CAST(SUBSTR(p.timestamp, 12, 2) AS INTEGER) AS hour_of_day,
    p.likes,
    p.shares,
    p.comments,
    (p.likes + p.shares + p.comments)     AS total_engagement,
    p.text_content,
    p.likes_is_outlier,
    p.shares_is_outlier,
    p.comments_is_outlier
FROM posts p
WHERE p.likes IS NOT NULL;

CREATE VIEW vw_user_performance AS
SELECT
    p.user_id,
    u.location,
    u.language,
    u.follower_count,
    COUNT(p.post_id)                               AS total_posts,
    ROUND(AVG(p.likes), 1)                         AS avg_likes,
    ROUND(AVG(p.shares), 1)                        AS avg_shares,
    ROUND(AVG(p.comments), 1)                      AS avg_comments,
    ROUND(AVG(p.likes + p.shares + p.comments), 1) AS avg_total_engagement,
    COUNT(DISTINCT p.platform)                     AS platforms_used
FROM posts p
LEFT JOIN users u ON p.user_id = u.user_id
WHERE p.likes IS NOT NULL
GROUP BY p.user_id;

CREATE VIEW vw_platform_summary AS
SELECT
    platform,
    COUNT(*)                                       AS total_posts,
    COUNT(DISTINCT user_id)                        AS unique_users,
    ROUND(AVG(likes), 1)                           AS avg_likes,
    ROUND(AVG(shares), 1)                          AS avg_shares,
    ROUND(AVG(comments), 1)                        AS avg_comments,
    ROUND(AVG(likes + shares + comments), 1)       AS avg_total_engagement,
    ROUND(1.0 * COUNT(*) / COUNT(DISTINCT user_id), 2) AS posts_per_user
FROM posts
WHERE platform IS NOT NULL AND likes IS NOT NULL
GROUP BY platform;
""")
conn.commit()

print("=" * 55)
print("  SCHEMA — users table")
print("=" * 55)
for row in cursor.execute("PRAGMA table_info(users);"):
    pk = "  PRIMARY KEY" if row[5] else ""
    print(f"  {row[1]:<22} {row[2]:<12}{pk}")

print()
print("=" * 55)
print("  SCHEMA — posts table")
print("=" * 55)
for row in cursor.execute("PRAGMA table_info(posts);"):
    pk = "  PRIMARY KEY" if row[5] else ""
    print(f"  {row[1]:<22} {row[2]:<12}{pk}")

print()
print("=" * 55)
print("  VIEWS CREATED")
print("=" * 55)
views = pd.read_sql("SELECT name FROM sqlite_master WHERE type='view';", conn)
for v in views['name']:
    print(f"  {v}")

print()
print("Foreign Key: posts.user_id -> users.user_id")
print("All tables and views ready.")


  SCHEMA — users table
  user_id                TEXT        
  location               TEXT        
  language               TEXT        
  account_created        TEXT        
  follower_count         INTEGER     

  SCHEMA — posts table
  post_id                TEXT        
  user_id                TEXT        
  platform               TEXT        
  text_content           TEXT        
  timestamp              TEXT        
  likes                  REAL        
  shares                 INTEGER     
  comments               INTEGER     
  likes_is_outlier       INTEGER     
  shares_is_outlier      INTEGER     
  comments_is_outlier    INTEGER     
  user_in_registry       INTEGER     

  VIEWS CREATED
  vw_post_engagement
  vw_user_performance
  vw_platform_summary

Foreign Key: posts.user_id -> users.user_id
All tables and views ready.


In [4]:
schema_users = pd.DataFrame({
    'Column': ['user_id','location','language','account_created','follower_count'],
    'Type':   ['TEXT','TEXT','TEXT','TEXT','INTEGER'],
    'Key':    ['PRIMARY KEY','','','','']
})
schema_posts = pd.DataFrame({
    'Column': ['post_id','user_id','platform','text_content','timestamp','likes','shares','comments','likes_is_outlier','shares_is_outlier','comments_is_outlier','user_in_registry'],
    'Type':   ['TEXT','TEXT','TEXT','TEXT','TEXT','INT','INT','INT','INT','INT','INT','INT'],
    'Key':    ['PRIMARY KEY','FK -> users.user_id','','','','','','','','','','']
})
views = pd.DataFrame({'Views Created': ['vw_post_engagement','vw_user_performance','vw_platform_summary']})

save_output(
    [schema_users, schema_posts, views],
    ['users table', 'posts table', 'views'],
    r"d:\project\DataScience\Data Vortex\Phase2\ss1_schema.png",
    "Screenshot 1 - Schema Design"
)


Saved: d:\project\DataScience\Data Vortex\Phase2\ss1_schema.png


In [5]:
query = """
WITH monthly AS (
    SELECT year_month, COUNT(*) AS total_posts,
        ROUND(AVG(likes), 1) AS avg_likes,
        ROUND(AVG(shares), 1) AS avg_shares,
        ROUND(AVG(comments), 1) AS avg_comments,
        ROUND(AVG(total_engagement), 1) AS avg_total_engagement,
        SUM(total_engagement) AS monthly_total_engagement
    FROM vw_post_engagement WHERE year_month IS NOT NULL GROUP BY year_month
)
SELECT year_month, total_posts, avg_likes, avg_shares, avg_comments,
    avg_total_engagement,
    SUM(monthly_total_engagement) OVER (ORDER BY year_month) AS cumulative_engagement
FROM monthly ORDER BY year_month;
"""
df_trend = pd.read_sql(query, conn)
display(df_trend)


,year_month,total_posts,avg_likes,avg_shares,avg_comments,avg_total_engagement,cumulative_engagement
0,2024-05,853,2454.4,992.2,508.2,3954.7,3373354.0
1,2024-06,807,2428.6,985.1,505.2,3919.0,6535951.0
2,2024-07,829,2456.7,1011.6,483.4,3951.7,9811915.0
3,2024-08,854,2581.9,1009.4,491.3,4082.5,13298403.0
4,2024-09,791,2530.2,993.4,492.9,4016.5,16475453.0
5,2024-10,850,2542.9,1025.6,505.9,4074.4,19938669.0
6,2024-11,850,2513.4,1002.8,516.2,4032.4,23366228.0
7,2024-12,864,2516.4,1041.1,505.3,4062.8,26876491.0
8,2025-01,836,2411.1,1007.0,497.9,3916.1,30150319.0
9,2025-02,771,2454.7,1003.1,507.7,3965.4,33207678.0


In [6]:
save_output(
    [df_trend],
    ['Monthly Engagement with Cumulative Window Function'],
    r"d:\project\DataScience\Data Vortex\Phase2\ss2_trend.png",
    "Screenshot 2 - Trend Detection"
)


Saved: d:\project\DataScience\Data Vortex\Phase2\ss2_trend.png


In [7]:
query_summary = """
WITH stats AS (
    SELECT AVG(likes) AS mean_likes, AVG(shares) AS mean_shares, AVG(comments) AS mean_comments,
        AVG(likes*likes)-AVG(likes)*AVG(likes) AS var_likes,
        AVG(shares*shares)-AVG(shares)*AVG(shares) AS var_shares,
        AVG(comments*comments)-AVG(comments)*AVG(comments) AS var_comments
    FROM vw_post_engagement
),
z_scored AS (
    SELECT e.post_id, e.user_id, e.platform, e.date, e.likes, e.shares, e.comments, e.total_engagement,
        ROUND((e.likes-s.mean_likes)/SQRT(s.var_likes+1),2) AS z_likes,
        ROUND((e.shares-s.mean_shares)/SQRT(s.var_shares+1),2) AS z_shares,
        ROUND((e.comments-s.mean_comments)/SQRT(s.var_comments+1),2) AS z_comments
    FROM vw_post_engagement e, stats s
),
flagged AS (
    SELECT *,
        CASE WHEN z_likes>1.5 AND z_shares>1.5 THEN 'Viral Post'
             WHEN z_likes>1.5 THEN 'High Likes'
             WHEN z_shares>1.5 THEN 'High Shares'
             WHEN z_comments>1.5 THEN 'High Comments'
             WHEN z_likes<-1.5 THEN 'Low Likes Outlier'
             ELSE 'Other' END AS anomaly_type
    FROM z_scored
    WHERE ABS(z_likes)>1.5 OR ABS(z_shares)>1.5 OR ABS(z_comments)>1.5
)
SELECT anomaly_type, COUNT(*) AS anomaly_count,
    ROUND(AVG(likes),0) AS avg_likes, ROUND(AVG(shares),0) AS avg_shares,
    ROUND(AVG(z_likes),3) AS avg_z_likes, ROUND(AVG(z_shares),3) AS avg_z_shares
FROM flagged GROUP BY anomaly_type ORDER BY anomaly_count DESC;
"""

query_detail = """
WITH stats AS (
    SELECT AVG(likes) AS mean_likes, AVG(shares) AS mean_shares, AVG(comments) AS mean_comments,
        AVG(likes*likes)-AVG(likes)*AVG(likes) AS var_likes,
        AVG(shares*shares)-AVG(shares)*AVG(shares) AS var_shares,
        AVG(comments*comments)-AVG(comments)*AVG(comments) AS var_comments
    FROM vw_post_engagement
),
z_scored AS (
    SELECT e.post_id, e.user_id, e.platform, e.date, e.likes, e.shares, e.comments, e.total_engagement,
        ROUND((e.likes-s.mean_likes)/SQRT(s.var_likes+1),2) AS z_likes,
        ROUND((e.shares-s.mean_shares)/SQRT(s.var_shares+1),2) AS z_shares,
        ROUND((e.comments-s.mean_comments)/SQRT(s.var_comments+1),2) AS z_comments
    FROM vw_post_engagement e, stats s
)
SELECT * FROM z_scored
WHERE ABS(z_likes)>1.5 OR ABS(z_shares)>1.5
ORDER BY total_engagement DESC LIMIT 15;
"""

df_anom_summary = pd.read_sql(query_summary, conn)
df_anom_detail  = pd.read_sql(query_detail, conn)
display(df_anom_summary)
display(df_anom_detail)


,anomaly_type,anomaly_count,avg_likes,avg_shares,avg_z_likes,avg_z_shares
0,Other,1075,2506.0,527.0,0.008,-0.837
1,High Likes,626,4830.0,976.0,1.624,-0.056
2,High Shares,597,2309.0,1935.0,-0.129,1.612
3,Low Likes Outlier,566,174.0,919.0,-1.614,-0.154
4,High Comments,524,2456.0,914.0,-0.027,-0.163
5,Viral Post,43,4831.0,1933.0,1.625,1.609


,post_id,user_id,platform,date,likes,shares,comments,total_engagement,z_likes,z_shares,z_comments
0,ycjj5zzt7mvx,user_d9971ba6,Instagram,2025-02-19,4983.0,1919,991,7893.0,1.73,1.58,1.69
1,wo7py9aljg3t,user_o8le7hqf,Reddit,2025-04-28,4864.0,1981,948,7793.0,1.65,1.69,1.54
2,gmoeib832zbs,user_pe5yckyb,Facebook,2025-01-24,4902.0,1880,982,7764.0,1.67,1.52,1.66
3,5kvuyvf38nqx,user_z0feut2e,YouTube,2024-06-09,4923.0,1971,861,7755.0,1.69,1.67,1.24
4,pvfl3d8hj7jd,user_csluibwk,Instagram,2024-05-15,4989.0,1840,909,7738.0,1.73,1.45,1.40
5,tdgjjylpua20,user_8nvzxsuj,NaN,2024-06-02,4979.0,1932,812,7723.0,1.73,1.61,1.07
6,tne7s3o4l4wd,user_lr3fagdl,Instagram,2024-10-30,4931.0,1903,878,7712.0,1.69,1.56,1.30
7,a1kiwl618kzy,user_aaiari8o,Facebook,2025-04-09,4811.0,1952,920,7683.0,1.61,1.64,1.44
8,fp89q1ickn9w,user_h4lueh1i,Twitter,2025-03-15,4740.0,1933,955,7628.0,1.56,1.61,1.56
9,5n161ir5hhhr,user_u98jwp3f,YouTube,2025-01-26,4751.0,1981,878,7610.0,1.57,1.69,1.30


In [8]:
save_output(
    [df_anom_summary, df_anom_detail],
    ['Anomaly Summary by Type', 'Top 15 Anomalous Posts'],
    r"d:\project\DataScience\Data Vortex\Phase2\ss3_anomaly.png",
    "Screenshot 3 - Anomaly Discovery"
)


Saved: d:\project\DataScience\Data Vortex\Phase2\ss3_anomaly.png


In [10]:
query_seg = """
WITH segmented AS (
    SELECT user_id, location, language, follower_count, total_posts,
        avg_likes, avg_total_engagement, platforms_used,
        CASE WHEN avg_total_engagement>=8000 THEN 'Power Creator'
             WHEN avg_total_engagement>=5500 THEN 'High Performer'
             WHEN avg_total_engagement>=3000 THEN 'Moderate Engager'
             ELSE 'Low Activity' END AS segment,
        NTILE(4) OVER (ORDER BY avg_total_engagement) AS quartile,
        ROUND(PERCENT_RANK() OVER (ORDER BY avg_total_engagement)*100,1) AS percentile
    FROM vw_user_performance
)
SELECT segment, COUNT(*) AS users_in_segment,
    ROUND(AVG(total_posts),1) AS avg_posts,
    ROUND(AVG(avg_likes),1) AS avg_likes,
    ROUND(AVG(avg_total_engagement),1) AS avg_total_engagement,
    ROUND(AVG(follower_count),0) AS avg_followers,
    ROUND(AVG(platforms_used),2) AS avg_platforms_used,
    MIN(ROUND(percentile,1)) AS min_percentile,
    MAX(ROUND(percentile,1)) AS max_percentile
FROM segmented GROUP BY segment ORDER BY avg_total_engagement DESC;
"""

query_top = """
WITH ranked AS (
    SELECT user_id, location, language, follower_count, total_posts,
        avg_total_engagement, platforms_used,
        ROUND(PERCENT_RANK() OVER (ORDER BY avg_total_engagement)*100,1) AS percentile,
        NTILE(4) OVER (ORDER BY avg_total_engagement) AS quartile
    FROM vw_user_performance
)
SELECT * FROM ranked WHERE percentile>=95 ORDER BY avg_total_engagement DESC LIMIT 15;
"""

df_seg = pd.read_sql(query_seg, conn)
df_top = pd.read_sql(query_top, conn)
display(df_seg)
display(df_top)


,segment,users_in_segment,avg_posts,avg_likes,avg_total_engagement,avg_followers,avg_platforms_used,min_percentile,max_percentile
0,High Performer,22,3.0,3999.1,5886.8,25196.0,2.18,98.6,100.0
1,Moderate Engager,1357,6.9,2559.7,4092.7,24963.0,3.39,7.9,98.5
2,Low Activity,118,5.2,1329.3,2597.7,24902.0,2.97,0.0,7.8


,user_id,location,language,follower_count,total_posts,avg_total_engagement,platforms_used,percentile,quartile
0,user_ysbkmmxf,"Toronto, Canada",ja,46203,1,6994.0,1,100.0,4
1,user_h8y1m5k9,"Barcelona, Spain",en,38418,2,6757.5,2,99.9,4
2,user_swf9alie,"Dubai, UAE",es,23651,3,6509.0,2,99.9,4
3,user_py6fq7ii,"Delhi, India",de,18940,4,6387.5,2,99.8,4
4,user_jm2grmis,"Dubai, UAE",en,45842,3,6058.0,2,99.7,4
5,user_uw0vd87k,"Milan, Italy",hi,11640,3,6027.3,3,99.7,4
6,user_bx9xyj6k,"Shanghai, China",en,19618,2,5897.5,2,99.6,4
7,user_7ccsgn39,"Barcelona, Spain",zh,23684,1,5863.0,1,99.5,4
8,user_mcchsnr8,"Munich, Germany",ar,2602,5,5824.8,4,99.5,4
9,user_xjkv8d43,"Houston, USA",hi,13651,2,5783.5,1,99.4,4


In [11]:
save_output(
    [df_seg, df_top],
    ['User Segment Summary', 'Top Performers - 95th Percentile and Above'],
    r"d:\project\DataScience\Data Vortex\Phase2\ss4_segments.png",
    "Screenshot 4 - Behavioural Grouping"
)


Saved: d:\project\DataScience\Data Vortex\Phase2\ss4_segments.png


In [12]:
query_corr = """
WITH tiered AS (
    SELECT user_id, follower_count, avg_likes, avg_shares, avg_comments,
        avg_total_engagement, total_posts,
        CASE WHEN follower_count>=40000 THEN '4. Very High (40k+)'
             WHEN follower_count>=25000 THEN '3. High (25k-40k)'
             WHEN follower_count>=10000 THEN '2. Mid (10k-25k)'
             ELSE '1. Low (under 10k)' END AS follower_tier
    FROM vw_user_performance WHERE follower_count IS NOT NULL
)
SELECT follower_tier, COUNT(*) AS user_count,
    ROUND(AVG(follower_count),0) AS avg_followers,
    ROUND(AVG(avg_likes),1) AS avg_likes,
    ROUND(AVG(avg_shares),1) AS avg_shares,
    ROUND(AVG(avg_comments),1) AS avg_comments,
    ROUND(AVG(avg_total_engagement),1) AS avg_total_engagement
FROM tiered GROUP BY follower_tier ORDER BY follower_tier;
"""

query_lang = """
SELECT language, COUNT(DISTINCT user_id) AS unique_users,
    ROUND(AVG(avg_likes),1) AS avg_likes,
    ROUND(AVG(avg_shares),1) AS avg_shares,
    ROUND(AVG(avg_comments),1) AS avg_comments,
    ROUND(AVG(avg_total_engagement),1) AS avg_total_engagement
FROM vw_user_performance GROUP BY language ORDER BY avg_total_engagement DESC;
"""

df_corr = pd.read_sql(query_corr, conn)
df_lang  = pd.read_sql(query_lang, conn)
display(df_corr)
display(df_lang)


,follower_tier,user_count,avg_followers,avg_likes,avg_shares,avg_comments,avg_total_engagement
0,1. Low (under 10k),287,5209.0,2434.6,1037.5,506.8,3979.0
1,2. Mid (10k-25k),469,17463.0,2516.2,1011.6,505.9,4033.7
2,3. High (25k-40k),449,32544.0,2484.6,1000.6,503.7,3989.0
3,4. Very High (40k+),292,44759.0,2478.8,1008.4,502.4,3989.6


,language,unique_users,avg_likes,avg_shares,avg_comments,avg_total_engagement
0,en,153,2571.8,1034.3,503.5,4109.6
1,pt,143,2516.0,1036.0,506.6,4058.6
2,ar,143,2525.5,1010.2,500.2,4035.9
3,zh,167,2484.5,1028.3,496.4,4009.3
4,ja,156,2503.1,982.8,502.5,3988.4
5,ru,143,2460.7,1008.1,515.6,3984.5
6,fr,150,2475.5,974.6,525.9,3976.0
7,de,140,2461.9,1007.1,504.0,3972.9
8,es,146,2439.1,1041.9,486.1,3967.1
9,hi,156,2400.5,1003.7,507.3,3911.6


In [13]:
save_output(
    [df_corr, df_lang],
    ['Follower Count Tier vs Engagement', 'Language vs Engagement'],
    r"d:\project\DataScience\Data Vortex\Phase2\ss5_correlation.png",
    "Screenshot 5 - Correlation Analysis"
)


Saved: d:\project\DataScience\Data Vortex\Phase2\ss5_correlation.png


In [14]:
query_e3 = """
SELECT
    platform,
    ROUND(AVG(likes), 2)                         AS avg_likes,
    ROUND(AVG(shares), 2)                        AS avg_shares,
    ROUND(AVG(comments), 2)                      AS avg_comments,
    ROUND(AVG(likes + shares + comments), 2)     AS avg_total_engagement
FROM posts
WHERE platform IS NOT NULL
  AND likes IS NOT NULL
GROUP BY platform
ORDER BY avg_total_engagement DESC;
"""

df_e3 = pd.read_sql(query_e3, conn)
print("E3 - Average Engagement by Platform")
print("=" * 55)
display(df_e3)


E3 - Average Engagement by Platform


,platform,avg_likes,avg_shares,avg_comments,avg_total_engagement
0,YouTube,2531.90,1019.41,500.09,4051.40
1,Instagram,2499.26,1041.16,501.92,4042.33
2,Facebook,2522.82,984.93,501.49,4009.24
3,Reddit,2485.49,1002.35,508.32,3996.16
4,Twitter,2430.05,1009.27,509.87,3949.19


In [15]:
save_output(
    [df_e3],
    ['E3 - Average Engagement by Platform'],
    r"d:\project\DataScience\Data Vortex\Phase2\e3_platform_engagement.png",
    "Easy Level E3 - Average Engagement by Platform"
)


Saved: d:\project\DataScience\Data Vortex\Phase2\e3_platform_engagement.png


In [16]:
query_m2 = """
WITH user_engagement AS (
    SELECT
        p.user_id,
        u.follower_count,
        CASE
            WHEN u.follower_count >= 25000 THEN 'High Follower (25k+)'
            ELSE 'Low Follower (under 25k)'
        END AS follower_group,
        COUNT(p.post_id)                                AS post_count,
        ROUND(AVG(p.likes + p.shares + p.comments), 2) AS avg_engagement_per_post,
        SUM(p.likes + p.shares + p.comments)            AS total_engagement
    FROM posts p
    JOIN users u ON p.user_id = u.user_id
    WHERE p.likes IS NOT NULL
      AND u.follower_count IS NOT NULL
    GROUP BY p.user_id
)
SELECT
    follower_group,
    COUNT(*)                                    AS user_count,
    ROUND(AVG(follower_count), 0)               AS avg_follower_count,
    ROUND(AVG(avg_engagement_per_post), 2)      AS avg_engagement_per_post,
    ROUND(AVG(total_engagement), 0)             AS avg_total_engagement,
    ROUND(AVG(post_count), 1)                   AS avg_posts_per_user
FROM user_engagement
GROUP BY follower_group
ORDER BY avg_engagement_per_post DESC;
"""

df_m2 = pd.read_sql(query_m2, conn)
print("M2 - High Follower vs Low Follower User Engagement")
print("=" * 55)
display(df_m2)


M2 - High Follower vs Low Follower User Engagement


,follower_group,user_count,avg_follower_count,avg_engagement_per_post,avg_total_engagement,avg_posts_per_user
0,Low Follower (under 25k),756,12811.0,4012.95,26932.0,6.7
1,High Follower (25k+),741,37358.0,3989.22,26454.0,6.6


In [17]:
save_output(
    [df_m2],
    ['M2 - High Follower vs Low Follower Engagement Comparison'],
    r"d:\project\DataScience\Data Vortex\Phase2\m2_follower_engagement.png",
    "Medium Level M2 - Do High Follower Users Get More Engagement?"
)


Saved: d:\project\DataScience\Data Vortex\Phase2\m2_follower_engagement.png


In [ ]:

df_corrupted = pd.read_csv(
    r"d:\project\DataScience\Data Vortex\Dataset\Social_Engine_Posts_Corrupted.csv",
    engine='python',
    on_bad_lines='warn',
    quotechar='"',
    skipinitialspace=True
)

df_corrupted.to_sql("posts_raw", conn, if_exists="replace", index=False)
conn.commit()

print(f"Corrupted posts loaded into posts_raw: {len(df_corrupted)} rows")
display(df_corrupted.head(3))


Corrupted posts loaded into posts_raw: 12360 rows


,post_id,user_id,platform,text_content,timestamp,likes,shares,comments
0,to64mgey2v3y,user_vfxs1pry,Reddit,Bummed out with my new Air Max from Nike! Abso...,25-09-2024,4488.0,1456,673
1,7f0wdauzbj89,user_8l7rv5oe,Reddit,My one month review of Pepsi Crystal Pepsi: Hi...,1722528840,789.0,1484,39
2,dvvhg8eel45x,user_nfo3ih5u,NaN,Just unboxed my new Highlander from Toyota. Ex...,2025-04-13T20:12:18,NaN,1410,839


In [19]:
query_h5_detail = """
WITH anomalies AS (
    SELECT post_id, 'Negative Likes'     AS anomaly_type FROM posts_raw
    WHERE CAST(likes AS REAL) < 0

    UNION ALL

    SELECT post_id, 'Missing Platform'   AS anomaly_type FROM posts_raw
    WHERE platform IS NULL
       OR TRIM(CAST(platform AS TEXT)) = ''
       OR UPPER(TRIM(CAST(platform AS TEXT))) = 'NULL'

    UNION ALL

    SELECT post_id, 'Missing Text'       AS anomaly_type FROM posts_raw
    WHERE text_content IS NULL
       OR TRIM(CAST(text_content AS TEXT)) = ''

    UNION ALL

    SELECT post_id, 'HTML Corruption'    AS anomaly_type FROM posts_raw
    WHERE CAST(text_content AS TEXT) LIKE '%&amp;%'
       OR CAST(text_content AS TEXT) LIKE '%<div>%'
       OR CAST(text_content AS TEXT) LIKE '%<br>%'
       OR CAST(text_content AS TEXT) LIKE '%&lt;%'
       OR CAST(text_content AS TEXT) LIKE '%&gt;%'
)
SELECT post_id, anomaly_type
FROM anomalies
ORDER BY post_id, anomaly_type;
"""

query_h5_summary = """
WITH anomalies AS (
    SELECT post_id, 'Negative Likes'     AS anomaly_type FROM posts_raw
    WHERE CAST(likes AS REAL) < 0

    UNION ALL

    SELECT post_id, 'Missing Platform'   AS anomaly_type FROM posts_raw
    WHERE platform IS NULL
       OR TRIM(CAST(platform AS TEXT)) = ''
       OR UPPER(TRIM(CAST(platform AS TEXT))) = 'NULL'

    UNION ALL

    SELECT post_id, 'Missing Text'       AS anomaly_type FROM posts_raw
    WHERE text_content IS NULL
       OR TRIM(CAST(text_content AS TEXT)) = ''

    UNION ALL

    SELECT post_id, 'HTML Corruption'    AS anomaly_type FROM posts_raw
    WHERE CAST(text_content AS TEXT) LIKE '%&amp;%'
       OR CAST(text_content AS TEXT) LIKE '%<div>%'
       OR CAST(text_content AS TEXT) LIKE '%<br>%'
       OR CAST(text_content AS TEXT) LIKE '%&lt;%'
       OR CAST(text_content AS TEXT) LIKE '%&gt;%'
)
SELECT
    anomaly_type,
    COUNT(*)    AS affected_posts,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM posts_raw), 2) AS pct_of_total
FROM anomalies
GROUP BY anomaly_type
ORDER BY affected_posts DESC;
"""

df_h5_summary = pd.read_sql(query_h5_summary, conn)
df_h5_detail  = pd.read_sql(query_h5_detail, conn)

print("H5 - Data Anomaly Detection on Original Corrupted Dataset")
print("=" * 60)
print("Summary by Anomaly Type:")
display(df_h5_summary)
print(f"\nTotal anomalous records: {len(df_h5_detail)}")
print("\nSample anomalous posts:")
display(df_h5_detail.head(20))


H5 - Data Anomaly Detection on Original Corrupted Dataset
Summary by Anomaly Type:


,anomaly_type,affected_posts,pct_of_total
0,Missing Platform,1846,14.94
1,Missing Text,1746,14.13
2,HTML Corruption,1004,8.12
3,Negative Likes,525,4.25



Total anomalous records: 5121

Sample anomalous posts:


,post_id,anomaly_type
0,003s4ulm32tk,HTML Corruption
1,005g54tmt26m,Negative Likes
2,0066x8nnmouc,HTML Corruption
3,0066x8nnmouc,Missing Platform
4,00pk8aa72o8x,Missing Text
5,00u9otx16xfc,Missing Platform
6,014e8jqloj6h,Missing Text
7,01gbk9id4v75,Missing Text
8,01kgwhi645er,Missing Platform
9,01kgwhi645er,Negative Likes


In [20]:
save_output(
    [df_h5_summary, df_h5_detail.head(20)],
    ['Anomaly Summary by Type', 'Sample of Anomalous Posts (first 20)'],
    r"d:\project\DataScience\Data Vortex\Phase2\h5_data_anomalies.png",
    "Hard Level H5 - Identify Data Anomalies in Corrupted Dataset"
)


Saved: d:\project\DataScience\Data Vortex\Phase2\h5_data_anomalies.png
